# Jev hallucination judgments on saved TriviaQA answers

This is a separate notebook. It reuses the saved Qwen predictions and BLEURT-20 scores from the completed LAYA run; it does not regenerate answers or recalculate BLEURT. Jev sees only the question, predicted answer, and the supplied assistant policy—not any LAYA predictions. The three Jev questions are the exact supplied `noul`, `choice`, and `score` definitions. Enable **Internet**; a GPU is not required. Attach notebook output `nvthaai/laya-triviaqa-gpu-multijudge` and add a notebook secret named `TYPESAFE_API_KEY` (do not paste the key into a cell). Each Jev request may be billable.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!pip install -q typesafe-sdk

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("TYPESAFE_API_KEY")
if not key:
    raise RuntimeError("Missing Kaggle secret TYPESAFE_API_KEY")
os.environ["TYPESAFE_API_KEY"] = key
del key
print("Jev key loaded from Kaggle Secrets")

In [ ]:
from pathlib import Path
import subprocess, sys

source_csvs = [
    path for path in Path("/kaggle/input").rglob("laya_triviaqa_gpu_multijudge/results.csv")
    if "laya-triviaqa-gpu-multijudge" in path.parts
]
if len(source_csvs) != 1:
    raise RuntimeError(f"Expected one completed LAYA result CSV, found: {source_csvs}")
SOURCE_CSV = source_csvs[0]
OUTPUT_DIR = Path("/kaggle/working/jev_triviaqa_multijudge")
MAX_SAMPLES = 0  # 0 = all saved TriviaQA rows; set to 1 for an initial paid API smoke test
command = [
    sys.executable, "/kaggle/working/SALA/jev_triviaqa.py",
    "--source-csv", str(SOURCE_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--max-samples", str(MAX_SAMPLES),
]
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / "results.csv")
display(results.head())
print(json.dumps(json.loads((OUTPUT_DIR / "summary.json").read_text()), indent=2))
print(f"Saved {len(results):,} rows to {OUTPUT_DIR}")